# 03 — Baseline Models

This notebook trains baseline classifiers for `Crime Solved` and compares them on the validation set:
- Logistic Regression
- Random Forest
- XGBoost
- LightGBM

It saves the best-performing baseline pipeline under `models/`.


In [3]:
# Setup + load engineered splits
from pathlib import Path
import sys

import pandas as pd

_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import DataPaths, load_processed_split
from src.train import TrainConfig, dataframe_X_y, fit_pipeline, make_pipeline, save_model
from src.evaluate import evaluate_binary, plot_confusion, plot_roc_curve

paths = DataPaths.from_project_root(PROJECT_ROOT)

train_df = load_processed_split(paths.processed_dir, "train")
val_df = load_processed_split(paths.processed_dir, "val")
test_df = load_processed_split(paths.processed_dir, "test")

X_train, y_train = dataframe_X_y(train_df)
X_val, y_val = dataframe_X_y(val_df)
X_test, y_test = dataframe_X_y(test_df)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

cfg = TrainConfig(random_state=42, n_jobs=-1)
MODELS = ["logreg", "rf", "xgb", "lgb"]

FileNotFoundError: Could not find processed split 'train' in C:\Users\Wassim Rahali\Desktop\Crime IA\data\processed.

In [ ]:
# Train + validate
results = []
trained = {}

for name in MODELS:
    print("\n====", name, "====")
    pipe = make_pipeline(X_train, name, cfg=cfg)
    pipe = fit_pipeline(pipe, X_train, y_train)

    val_res = evaluate_binary(pipe, X_val, y_val)
    print("Val ROC-AUC:", round(val_res.roc_auc, 4))
    print("Val F1 (pos=1):", round(val_res.report.get("1", {}).get("f1-score", 0.0), 4))

    results.append(
        {
            "model": name,
            "val_roc_auc": val_res.roc_auc,
            "val_f1_pos": val_res.report.get("1", {}).get("f1-score", 0.0),
        }
    )
    trained[name] = pipe

results_df = pd.DataFrame(results).sort_values("val_roc_auc", ascending=False)
results_df

In [ ]:
# Evaluate best model on test + save
best_name = results_df.iloc[0]["model"]
best_pipe = trained[best_name]

print("Best (by val ROC-AUC):", best_name)

test_res = evaluate_binary(best_pipe, X_test, y_test)
print("Test ROC-AUC:", round(test_res.roc_auc, 4))
print(pd.DataFrame(test_res.report).T[["precision", "recall", "f1-score", "support"]])

plot_confusion(test_res.confusion, title=f"Confusion matrix — {best_name} (test)")
plot_roc_curve(best_pipe, X_test, y_test, title=f"ROC — {best_name} (test)")

# Save pipeline
model_path = PROJECT_ROOT / "models" / f"{best_name}_baseline.joblib"
save_model(best_pipe, model_path)
print("Saved:", model_path)

# Bias audit quick check: ROC-AUC by Victim Race (if enough samples)
from src.evaluate import group_metrics_by_column, predict_proba_positive

proba_test = predict_proba_positive(best_pipe, X_test)
audit_df = test_df.reset_index(drop=True)

by_race = group_metrics_by_column(
    df_with_target=audit_df,
    y_true=y_test,
    y_proba=proba_test,
    group_col="Victim Race",
    min_group_size=5000,
)
by_race.head(15)